In [1]:
import pandas as pd

In [2]:
df=pd.read_csv("C:/Users/300408947/Downloads/all_stocks_5yr/all_stocks_5yr.csv")
df.head()

,date,open,high,low,close,volume,name
0,2013-02-08,15.07,15.12,14.63,14.75,8407500,AAL
1,2013-02-11,14.89,15.01,14.26,14.46,8882000,AAL
2,2013-02-12,14.45,14.51,14.10,14.27,8126000,AAL
3,2013-02-13,14.30,14.94,14.25,14.66,10259500,AAL
4,2013-02-14,14.94,14.96,13.16,13.99,31879900,AAL


In [3]:
df_10x=pd.concat([df]*10,ignore_index=True)
df_100x=pd.concat([df]*100,ignore_index=True)

In [4]:
df.to_csv("stocks_1x.csv",index=False)
df_10x.to_csv("stocks_10x.csv",index=False)
df_100x.to_csv("stocks_100x.csv",index=False)

In [5]:
df.to_parquet("stocks_1x.parquet")
df_10x.to_parquet("stocks_10x.parquet")
df_100x.to_parquet("stocks_100x.parquet")

In [7]:
import os
files=[
    "stocks_1x.csv","stocks_1x.parquet",
    "stocks_10x.csv","stocks_10x.parquet",
    "stocks_100x.csv","stocks_100x.parquet"
]

for f in files:
    print(f, round(os.path.getsize(f)/1024/1024,2),"MB")

stocks_1x.csv 28.8 MB
stocks_1x.parquet 10.03 MB
stocks_10x.csv 288.01 MB
stocks_10x.parquet 94.92 MB
stocks_100x.csv 2880.05 MB
stocks_100x.parquet 947.92 MB


In [9]:
import time 
start=time.time()
pd.read_csv("stocks_100x.csv")
print("CSV:",time.time()-start)

start=time.time()
pd.read_parquet("stocks_100x.parquet")
print("Parquet:",time.time()-start)

CSV: 51.990694999694824
Parquet: 8.841989755630493


In [10]:
for f in files:
    os.remove(f)

In [12]:
# Part 2
start=time.time()
df_pd=pd.read_csv("C:/Users/300408947/Downloads/all_stocks_5yr/all_stocks_5yr.csv")
print("Pandas Load:",time.time()-start)

Pandas Load: 0.48967552185058594


In [13]:
import polars as pl
start=time.time()
df_pl=pl.read_csv("C:/Users/300408947/Downloads/all_stocks_5yr/all_stocks_5yr.csv")
print("Polars load: ",time.time()-start)

Polars load:  0.20959115028381348


In [15]:
df_pd["SMA20"]=df_pd.groupby("name")["close"].transform(
    lambda x: x.rolling(20).mean()
)
df_pd["PriceChange"]=df_pd["close"].diff()

In [16]:
df_pd["NextClose"]=df_pd.groupby("name")["close"].shift(-1)
df_pd=df_pd.dropna()

In [19]:
# Train model 1 Linear Regression
from sklearn.linear_model import LinearRegression
features=["open","high","low","volume","SMA20","PriceChange"]
X=df_pd[features]
y=df_pd["NextClose"]
split=int(len(df_pd)*0.8)

X_train,X_test=X[:split],X[split:]
y_train,y_test=y[:split],y[split:]

model1=LinearRegression()
model1.fit(X_train,y_train)

pred1=model1.predict(X_test)

In [20]:
# Train model 2 Random Forest
from sklearn.ensemble import RandomForestRegressor
model2=RandomForestRegressor(n_estimators=50)
model2.fit(X_train,y_train)

pred2=model2.predict(X_test)

In [22]:
from sklearn.metrics import mean_absolute_error

print("Linear MAE;", mean_absolute_error(y_test,pred1))
print("RF MAE:",mean_absolute_error(y_test,pred2))

Linear MAE; 0.8229254839760352
RF MAE: 0.8612188288569387


In [24]:
results=df_pd.iloc[split:].copy()
results["Prediction"]=pred2
results.to_csv("C:/Users/300408947/Downloads/predictions.csv",index=False)